# Tidy data plotting

> **Krisk 0.9 notebook mode**
>
> This notebook runs locally with pandas and Krisk only. It does not require the Krisk
> server, MCP, SQLite, or PostgreSQL. Chart output embeds its renderer and can be used
> offline. The typed `ChartSpec` API is recommended for new work; legacy `krisk.plot`
> calls remain supported for existing notebooks.

`line_tidy` and `bar_tidy` are compatibility helpers for already-aggregated data.
They assume a two-dimensional DataFrame with a simple index and one series per column.
For new portable charts, long-form data plus `ChartSpec` is usually clearer.

In [ ]:
from pathlib import Path

import pandas as pd
import krisk.plot as kk
from krisk import Chart, ChartSpec, __version__

ROOT = Path.cwd()
if not (ROOT / "krisk" / "tests" / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "krisk" / "tests" / "data"

print(f"Krisk {__version__}")

## Wide time-series data

In [ ]:
prices = pd.read_csv(
    DATA_DIR / "stock_px.csv",
    parse_dates=True,
    index_col=0,
)
prices = prices[["AAPL", "MSFT", "XOM"]].resample("B").ffill()
prices.head()

In [ ]:
kk.line_tidy(prices)

A Series becomes a single-series tidy chart.

In [ ]:
kk.line_tidy(prices["AAPL"])

Stacked and normalized area charts remain supported.

In [ ]:
kk.line_tidy(prices, area=True, stacked=True)

In [ ]:
kk.line_tidy(prices, area=True, stacked=True, full=True)

## Tidy bars

In [ ]:
gapminder = pd.read_csv(DATA_DIR / "gapminderDataFiveYear.txt", sep="\t")
life_by_continent = gapminder.pivot_table(
    values="lifeExp",
    index="year",
    columns="continent",
    aggfunc="mean",
)
life_by_continent.head()

In [ ]:
kk.bar_tidy(life_by_continent)

In [ ]:
kk.bar_tidy(life_by_continent, stacked=True, trendline=True, annotate=True)

## Recommended long-form equivalent

In [ ]:
life_long = life_by_continent.reset_index().melt(
    id_vars="year",
    var_name="continent",
    value_name="life_expectancy",
)

Chart.from_dataframe(
    life_long,
    ChartSpec(
        kind="line",
        x="year",
        y="life_expectancy",
        color="continent",
        aggregate="mean",
        title="Life expectancy by continent",
        smooth=True,
    ),
)